# Inference Acceleration: Why Is Your LLM So Slow?

> **Background**: you ask an LLM a question and it streams tokens one by one. Why can't it output everything at once? Why is it sometimes fast and sometimes slow?
>
> Goal for this part: **understand the real root causes of slow inference, and how KV Cache, FlashAttention, and vLLM speed things up.**

There is one core issue:
**during autoregressive decoding, every new token would naively re-compute attention over all previous tokens.**
It is like re-reading the whole essay from the beginning after writing each new word.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)

### 1. The root cause: repeated computation

Autoregressive generation:

```
Step 1: input [BOS]                    -> attention -> predict "I"
Step 2: input [BOS, I]                 -> attention -> predict "love"
Step 3: input [BOS, I, love]           -> attention -> predict "you"
Step 4: input [BOS, I, love, you]      -> attention -> predict EOS
```

Problem: in Step 2 you re-compute K and V for `[BOS]` again.
In Step 3 you re-compute everything from Step 1+2 again.

```
Step 1: compute K1,V1 (1 token)
Step 2: compute K1,V1,K2,V2 (2 tokens; K1,V1 are repeated)
Step 3: compute K1,V1,K2,V2,K3,V3 (3 tokens; first 4 are repeated)
...
```

Generating N tokens costs about `1+2+...+N = O(N^2)` attention work.
For N=100, that's 5050 "token-attention" computations, not 100.


In [2]:
# Visualize repeated computation in naive autoregressive decoding
def naive_generation_cost(n_tokens):
    """Total token-attention work for naive autoregressive generation (in token units)."""
    return sum(range(1, n_tokens + 1))

for n in [10, 50, 100, 500]:
    cost = naive_generation_cost(n)
    print(
        f"Generate {n:3d} tokens -> compute attention for {cost:6d} token-steps "
        f"-> wasted {cost - n:5d} token-steps"
    )

print()
print(f"For 100 tokens, wasted {naive_generation_cost(100) - 100} extra computations!")
print("This is a root cause of slow LLM inference.")


Generate  10 tokens -> compute attention for     55 token-steps -> wasted    45 token-steps
Generate  50 tokens -> compute attention for   1275 token-steps -> wasted  1225 token-steps
Generate 100 tokens -> compute attention for   5050 token-steps -> wasted  4950 token-steps
Generate 500 tokens -> compute attention for 125250 token-steps -> wasted 124750 token-steps

For 100 tokens, wasted 4950 extra computations!
This is a root cause of slow LLM inference.


### 2. Solution 1: KV Cache (store what you already computed)

Core idea: K and V for previous tokens are already computed. Store them and reuse them.

```
Step 1: compute K1,V1 -> store in cache
Step 2: compute only K2,V2 -> read K1,V1 from cache -> concatenate
Step 3: compute only K3,V3 -> read previous K,V from cache -> concatenate
...
```

Effect: total compute drops from `O(N^2)` to `O(N)`.

```
Naive:    1+2+...+100 = 5050
KV cache: 1+1+...+1   = 100   (about 50x fewer)
```


In [3]:
class AttentionWithKVCache(nn.Module):
    """Self-attention with KV cache.

    Key idea:
    - First call (prefill): process the whole prompt, compute all K/V and store them.
    - Later calls (decode): compute K/V only for the new token and reuse cached K/V.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

        # KV cache storage
        self.k_cache = None
        self.v_cache = None

    def reset_cache(self):
        """Clear cache (call before starting a new generation)."""
        self.k_cache = None
        self.v_cache = None

    def forward(self, x, use_cache=True):
        """Forward.

        Input: x shape = [batch, seq_len, d_model]

        If use_cache=True and cache is not empty:
            x contains only the new token (seq_len=1)
            reuse cached K/V for old tokens and compute K/V for the new token
        """
        batch_size, seq_len, _ = x.shape

        # Compute Q, K, V
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        if use_cache:
            if self.k_cache is not None:
                # Concatenate old K/V with new K/V
                K = torch.cat([self.k_cache, K], dim=2)
                V = torch.cat([self.v_cache, V], dim=2)
            # Update cache
            self.k_cache = K
            self.v_cache = V

        # Attention computation (same math)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_out = torch.matmul(attn_probs, V)

        # Merge heads
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_O(attn_out)


In [4]:
# Compare: with KV cache vs without KV cache
attn = AttentionWithKVCache(d_model=64, num_heads=4)

# Simulate generating 20 tokens
N = 20

# No-cache version (naive)
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=False)
    sequence = torch.cat([sequence, torch.randn(1, 1, 64)], dim=1)
no_cache_time = time.time() - start

# With-cache version
attn.reset_cache()
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=True)
    sequence = torch.randn(1, 1, 64)  # only feed the new token each time
cache_time = time.time() - start

print(f"Without KV cache: {no_cache_time*1000:.1f} ms")
print(f"With KV cache:    {cache_time*1000:.1f} ms")
print(f"Speedup:          {no_cache_time/cache_time:.1f}x")
print()
print("Note: N=20 is small. At N=1000, the difference becomes huge.")


Without KV cache: 18.1 ms
With KV cache:    17.8 ms
Speedup:          1.0x

Note: N=20 is small. At N=1000, the difference becomes huge.


### 3. The memory cost of KV Cache

KV cache fixes compute, but introduces a **memory problem**.

A rough estimate (FP16):

```
KV cache per token ~= 2 * num_layers * d_model * 2 bytes

LLaMA-7B: 32 layers * 4096 dims * 2 * 2 bytes  ~= 1 MB / token
  2048 tokens -> ~2 GB KV cache

LLaMA-70B: 80 layers * 8192 dims * 2 * 2 bytes ~= 5 MB / token
  2048 tokens -> ~10 GB KV cache
```

And each request needs its own KV cache. 100 concurrent users * 2GB = 200GB VRAM.

This is a core problem vLLM is designed to address.


### 4. vLLM's key idea: PagedAttention

Problem: KV cache is usually allocated as one big contiguous chunk per request. But different requests produce different lengths, which leads to **fragmentation**.
Like a parking lot with cars of different sizes leaving unusable gaps.

vLLM's idea: split KV cache into fixed-size "pages" (blocks), managed like virtual memory.

```
Traditional (contiguous):
+------------------------------+
| request A KV (500 tokens)    |
+------------------------------+
| wasted fragment               |
+--------------+
| req B (100)  |
+--------------+

PagedAttention (paged):
+--+--+--+--+--+--+--+--+
|A1|B1|A2|C1|B2|A3|C2|A4|  each page fixed size (e.g. 16 tokens)
+--+--+--+--+--+--+--+--+
non-contiguous but far less fragmentation
```

Effect:
- memory utilization improves from ~20-40% to near 100%
- same VRAM can serve 2-4x more concurrent requests


In [5]:
# Intuition: contiguous allocation vs paged allocation memory waste
import random

def simulate_memory(requests, block_size=16):
    """Simulate wasted space under two allocation strategies."""

    # Contiguous allocation with alignment
    contiguous_wasted = 0
    for req_len in requests:
        aligned = ((req_len + 63) // 64) * 64
        contiguous_wasted += (aligned - req_len)

    # Paged allocation: waste only the last page slack
    paged_wasted = 0
    for req_len in requests:
        paged_wasted += (block_size - (req_len % block_size)) % block_size

    return contiguous_wasted, paged_wasted

# Simulate 50 random-length requests
random.seed(42)
requests = [random.randint(10, 500) for _ in range(50)]

cw, pw = simulate_memory(requests)
total = sum(requests)
print(f"50 requests, total tokens: {total}")
print(f"Contiguous waste: {cw} token slots ({cw/total*100:.1f}%)")
print(f"Paged waste:      {pw} token slots ({pw/total*100:.1f}%)")
print()
print(f"Paged allocation saves {cw - pw} token slots of memory!")


50 requests, total tokens: 11486
Contiguous waste: 1570 token slots (13.7%)
Paged waste:      418 token slots (3.6%)

Paged allocation saves 1152 token slots of memory!


### 5. FlashAttention: make attention compute faster

KV cache reduces *how much* you compute.
FlashAttention reduces *how slow* attention is, by fixing memory access.

Problem: the bottleneck is often not FLOPs, but **HBM (VRAM) bandwidth**.

Standard attention does multiple read/write passes to VRAM:
1. QK^T -> write to HBM
2. softmax -> read, compute, write
3. multiply by V -> read, compute, write

FlashAttention idea:
- tile Q,K,V into blocks
- stream blocks from HBM into on-chip SRAM
- compute attention fully in SRAM
- write only the final output back to HBM
- do NOT write intermediate matrices to HBM

Effect:
- 2-4x speedup
- 10-20x less VRAM usage for intermediates
- mathematically exact, not an approximation


In [6]:
# Intuition: VRAM bandwidth vs on-chip bandwidth (rough numbers)
print("=== Speed of different GPU memory levels (rough) ===")
print()
print("HBM (VRAM):  ~1.5 TB/s bandwidth, far from compute")
print("SRAM (on-chip): ~19 TB/s bandwidth, close to compute")
print()
print("How much HBM traffic does standard attention create?")
print("  QK^T matrix: seq_len^2 * 2 bytes (FP16)")
print("  For 2048 tokens: 2048^2 * 2 ~= 8 MB")
print("  For 8192 tokens: 8192^2 * 2 ~= 128 MB (huge)")
print()
print("FlashAttention avoids writing this matrix back to HBM.")
print("That is a big part of why FlashAttention is fast.")


=== Speed of different GPU memory levels (rough) ===

HBM (VRAM):  ~1.5 TB/s bandwidth, far from compute
SRAM (on-chip): ~19 TB/s bandwidth, close to compute

How much HBM traffic does standard attention create?
  QK^T matrix: seq_len^2 * 2 bytes (FP16)
  For 2048 tokens: 2048^2 * 2 ~= 8 MB
  For 8192 tokens: 8192^2 * 2 ~= 128 MB (huge)

FlashAttention avoids writing this matrix back to HBM.
That is a big part of why FlashAttention is fast.


### 6. Two phases of inference: Prefill vs Decode

LLM inference has two phases with very different bottlenecks:

| | Prefill | Decode |
|:---|:---|:---|
| What | process the full prompt once | generate new tokens one by one |
| Input length | long (thousands) | short (1 token per step) |
| Bottleneck | **compute** | **memory bandwidth** (reading KV cache) |
| Main optimizations | FlashAttention | KV cache + PagedAttention |
| Parallelism | high (tokens parallel) | low (serial) |

This is why the first token is often slow (prefill), and later tokens are faster (decode + KV cache).


### 7. Other acceleration techniques (at a glance)

| Technique | What | Speedup | Accuracy loss |
|:---|:---|:---|:---|
| **KV Cache** | reuse past K,V | 10-50x | none |
| **FlashAttention** | reduce HBM traffic | 2-4x | none |
| **PagedAttention (vLLM)** | paged KV management | 2-4x throughput | none |
| **Quantization (INT8/INT4)** | fewer bits for weights | 2-4x | small |
| **Speculative decoding** | draft+verify | 2-3x | none |
| **Continuous batching** | admit requests continuously | 5-10x throughput | none |
| **Tensor parallelism** | split layers across GPUs | ~#GPUs | none |

Speculative decoding is especially interesting; we cover it in the next part.


---

## Inference Acceleration Summary

1. [ok] Root cause: repeated K,V computation during autoregressive decoding -> O(N^2)
2. [ok] **KV cache**: store K,V -> O(N)
3. [ok] Memory issue: long sequences + concurrency -> KV cache explodes
4. [ok] **vLLM / PagedAttention**: paged KV cache -> reduce fragmentation -> 2-4x throughput
5. [ok] **FlashAttention**: compute in SRAM, avoid intermediate HBM writes -> 2-4x faster
6. [ok] Prefill (compute-bound) vs Decode (bandwidth-bound)
7. [ok] First token is slow (prefill), later tokens are faster (decode + KV cache)

One-sentence summary: slow inference is repeated compute -> KV cache reuses -> memory becomes the next limit -> vLLM pages it -> FlashAttention makes each attention step faster.
